In [37]:
# Client Setup
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

## Core Functions

### Chunking

In [38]:
# Chunk by section
import re

def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

### Embedding Generation

In [39]:
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list) # this is to handle both single string and list of strings input
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

### VectorIndex Implementation

- `VectorIndex` stores embedding vectors and their original documents together, keeping them in sync by index position.
- Each document must include a `"content"` field; `add_vector` validates this along with vector type and dimension.
- All vectors must share the same dimension - the first vector added sets the expected dimension for the store.
- `search(query, k)` accepts either:
  - a pre-computed numeric query vector (used in this notebook), or
  - a raw query string (if an `embedding_fn` was passed during initialization).
- The index computes distance from the query to **every stored vector** (brute-force linear scan), then sorts ascending and returns the top `k`.
- Supported distance metrics:
  - `cosine` distance = `1 - cosine_similarity` (default) - directional similarity, ignores magnitude
  - `euclidean` distance - straight-line distance in vector space
- **Smaller distance means more relevant** in both metrics.

**This notebook vs production:**
| | This notebook | Production (e.g. Pinecone, Qdrant, ChromaDB) |
|---|---|---|
| Search method | Brute-force linear scan O(n) | Approximate Nearest Neighbor (ANN) e.g. HNSW |
| Scale | Hundreds of vectors | Millions to billions of vectors |
| Persistence | In-memory only | Disk-backed, persistent |

> This notebook uses exact brute-force search, which is fine for learning but does not scale. Production systems use ANN algorithms like **HNSW** (Hierarchical Navigable Small World) to search billions of vectors in milliseconds.

In [40]:
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric must be 'cosine' or 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        if not self._embedding_fn:
            raise ValueError(
                "Embedding function not provided during initialization."
            )
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("Document 'content' must be a string.")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError(
                    "Embedding function not provided for string query."
                )
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError(
                "Query must be either a string or a list of numbers."
            )

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"Query vector dimension mismatch. Expected {self._vector_dim}, got {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k must be a positive integer.")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("Vector must be a list of numbers.")
        if not isinstance(document, dict):
            raise TypeError("Document must be a dictionary.")
        if "content" not in document:
            raise ValueError(
                "Document dictionary must contain a 'content' key."
            )

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"Inconsistent vector dimension. Expected {self._vector_dim}, got {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        if len(vec1) != len(vec2):
            raise ValueError("Vectors must have the same dimension")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def __len__(self) -> int:
        return len(self.vectors)

    def __repr__(self) -> str:
        has_embed_fn = "Yes" if self._embedding_fn else "No"
        return f"VectorIndex(count={len(self)}, dim={self._vector_dim}, metric='{self._distance_metric}', has_embedding_fn='{has_embed_fn}')"

In [41]:
with open("./assets/1_example_report.md", "r") as f:
    text = f.read()

## Test Implementation

**1. Chunk by Section**

In [42]:
chunks = chunk_by_section(text)

# print(chunks) # Print the first chunk - For demonstration purposes only

**2. Generate embeddings for each chunk**

In [43]:
chunk_embeddings = generate_embedding(chunks)

# chunk_embeddings # Print the embedding for the first chunk - For demonstration purposes only


**3. Create a vector store and add each embedding to it**
- Store Embeddings & Original Chunks together in VectorDB

In [44]:
store = VectorIndex()

# Pair/Zip the chunks and their corresponding embeddings together to add to the vector store
for chunk_embeddings, chunk in zip(chunk_embeddings, chunks):
    store.add_vector(vector=chunk_embeddings, document={"content": chunk})

**4. Convert User query into Embedding**

In [45]:
user_query = "What are the key findings from the report with regards to biomedical outcomes?" # Note: There is no word "biomedical outcomes" in the report, so this is to demonstrate the ability of the vector store to find relevant information even when there are no exact keyword matches.

user_query_embedding = generate_embedding(user_query, input_type="query")

# user_query_embedding # Print the embedding for the user query - For demonstration purposes only

**5. Compare Vector store with user embedding to find most relevant chunks**

In [46]:
# Find the 2 most relevant chunks
relevant_chunks = store.search(user_query_embedding, k=2)

for chunk, distance in relevant_chunks:
    print("Distance:", distance)
    print(chunk["content"][:200])  # Print the first 200 characters of each relevant chunk for demonstration purposes only

Distance: 0.5428585680741576
Section 1: Medical Research - Understanding XDR-471 Syndrome

This year saw significant strides in our understanding of XDR-471 syndrome, a rare neurodegenerative condition previously hampered by diag
Distance: 0.5713757929462627
Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update

Promising results emerged from the Phase IIa clinical trial (`Trial ID: CTX204b-P2A-001`) for Compound CTX-204b, our lead ca
